# Python Testing

---

## 1. Practical Experience Creating Tests

Common Python testing frameworks used in real projects:

- **pytest** – Most widely used. Clean syntax, powerful fixtures, rich plugin ecosystem.
- **unittest** – Built-in, follows xUnit style. Good for structured, class-based test suites.
- **nose2** – Successor to nose. Less common today but still found in legacy codebases.
- **hypothesis** – Property-based testing. Generates edge-case inputs automatically.

```python
# pytest example
def test_addition():
    assert 1 + 1 == 2
```

---

## 2. Test Types

| Type | Scope | Speed | Example |
|---|---|---|---|
| **Unit** | Single function/class | Fast | Test a `calculate_tax()` function |
| **Integration** | Multiple modules/services | Medium | Test DB write + read flow |
| **E2E (End-to-End)** | Full system/user flow | Slow | Simulate user login via browser |

- **Unit tests** validate isolated logic with no external dependencies.
- **Integration tests** verify that components work correctly together (e.g., service + database).
- **E2E tests** simulate real user scenarios across the full stack (tools: Selenium, Playwright, Cypress).

---

## 3. Testing Pyramid

```
        /\
       /E2E\          ← Few, slow, expensive
      /------\
     /  Integ  \      ← Some, medium cost
    /------------\
   /    Unit Tests \  ← Many, fast, cheap
  /________________\
```

The pyramid guides the **proportion** of test types:
- Most tests should be **unit tests** (fast, isolated, cheap to maintain).
- Fewer **integration tests** (validate contracts between components).
- Very few **E2E tests** (high value but fragile and slow).

Inverting the pyramid (too many E2E) leads to slow pipelines and brittle test suites.

---

## 4. Mock, Stub, Patch

### Definitions

| Concept | Purpose | When to use |
|---|---|---|
| **Mock** | Replaces an object and records calls | Verify interactions (was a method called?) |
| **Stub** | Returns predefined data | Control indirect inputs to the unit under test |
| **Patch** | Temporarily replaces something in scope | Replace external calls (HTTP, DB, file I/O) |

```python
from unittest.mock import MagicMock, patch

# Mock: verify a method was called
service = MagicMock()
service.send_email("user@test.com")
service.send_email.assert_called_once_with("user@test.com")

# Patch: replace external dependency
with patch("mymodule.requests.get") as mock_get:
    mock_get.return_value.status_code = 200
    result = fetch_data("https://api.example.com")
```

### Pros and Cons

| | Pros | Cons |
|---|---|---|
| **Mocking** | Isolates unit; fast tests | Overuse hides real bugs; tests may not reflect reality |
| **Patching** | Avoids side effects (I/O, network) | Tight coupling to implementation details |
| **Stubs** | Deterministic outputs | May not cover all real-world cases |

> ⚠️ **Golden rule**: Mock at the boundary. Don't mock what you own.

---

## 5. Common Problems with Tests

### Slow Execution
- **Cause**: Too many I/O calls, no parallelism, heavy setup/teardown.
- **Solution**: Use `pytest-xdist` for parallel runs; mock I/O; optimize fixtures with proper scope (`session`, `module`).

```bash
pytest -n auto  # run tests in parallel
```

### Flakiness
- **Cause**: Time dependencies, shared state, race conditions, network calls.
- **Solution**:
  - Use deterministic data (freeze time with `freezegun`).
  - Isolate test state (fresh DB per test).
  - Retry only as a last resort (`pytest-rerunfailures`).
  - Remove real network calls — use mocks or `responses` library.

```python
from freezegun import freeze_time

@freeze_time("2024-01-01")
def test_expiry():
    assert is_expired("2023-12-31") == True
```

---

## 6. TDD Approach (Test-Driven Development)

**Red → Green → Refactor** cycle:
1. Write a **failing test** (Red).
2. Write the **minimum code** to pass it (Green).
3. **Refactor** while keeping tests green.

```python
# Step 1 – failing test
def test_discount():
    assert apply_discount(100, 10) == 90

# Step 2 – minimum implementation
def apply_discount(price, pct):
    return price - (price * pct / 100)
```

### Pros and Cons

| Pros | Cons |
|---|---|
| Forces clear requirements before coding | Slower initial development |
| High test coverage naturally | Hard to apply on UI or legacy code |
| Better, decoupled design | Requires discipline and team buy-in |
| Confidence to refactor | Overhead for trivial/exploratory code |

---

## 7. Test Runners

### pytest (recommended)
- Simple syntax, no boilerplate.
- Fixtures, parametrize, plugins (`pytest-cov`, `pytest-xdist`, `pytest-mock`).
- Great CLI output.

```bash
pytest tests/ -v --cov=myapp
```

### nose2
- Compatible with `unittest`.
- Less actively maintained; avoid for new projects.

### IDE Configuration

**VS Code** – `.vscode/settings.json`:
```json
{
  "python.testing.pytestEnabled": true,
  "python.testing.pytestArgs": ["tests"]
}
```

**PyCharm** – Go to `Settings → Tools → Python Integrated Tools → Testing → pytest`.

---

## 8. Best Practices — What Makes a Good Test?

| Principle | Description |
|---|---|
| **Independent** | Tests don't depend on each other or shared state |
| **Atomic** | One assertion per test (one reason to fail) |
| **Maintainable** | Easy to read, update, and understand |
| **See it fail first** | Always verify the test catches a real failure (Red phase) |
| **Cyclomatic complexity = 1** | No `if/else` logic inside tests — just setup, act, assert |
| **Descriptive name** | `test_invoice_total_includes_tax` not `test_1` |
| **Fast** | Unit tests should run in milliseconds |
| **Deterministic** | Same result every time, no randomness |

```python
# Bad: multiple assertions, unclear name
def test_user():
    u = User("Alice")
    assert u.name == "Alice"
    assert u.is_active == True  # two concerns!

# Good: atomic and descriptive
def test_new_user_is_active_by_default():
    user = User("Alice")
    assert user.is_active == True
```

### Why can we trust tests?
Tests are trustworthy when they are **independent**, **seen to fail**, and **test behavior not implementation**. A test that was never seen to fail may be a false green.

---

## 9. Compare: unittest vs pytest

| Feature | unittest | pytest |
|---|---|---|
| Syntax | Class-based, verbose | Function-based, minimal |
| Built-in | ✅ Yes | ❌ Install separately |
| Fixtures | `setUp/tearDown` | Powerful `@pytest.fixture` |
| Parametrize | Manual | `@pytest.mark.parametrize` |
| Plugin ecosystem | Limited | Rich (100+ plugins) |
| Output readability | Basic | Excellent |
| Async support | Limited | Via `pytest-asyncio` |

```python
# unittest
import unittest
class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)

# pytest equivalent
def test_add():
    assert 1 + 1 == 2
```

> **Recommendation**: Use **pytest** for new projects. Use `unittest` only when bound to legacy code or a framework that requires it.

---

## 10. Tests as Quality Attribute Guards

Beyond functional correctness, tests validate **non-functional requirements**:

### Load & Performance Testing

| Tool | Type | Notes |
|---|---|---|
| **Locust** | Load testing | Python-based, scriptable, web UI |
| **Gatling** | Load/stress | Scala DSL, high performance, great reports |
| **tox** | Test environment manager | Run tests across multiple Python versions |

```python
# locust example
from locust import HttpUser, task

class APIUser(HttpUser):
    @task
    def get_products(self):
        self.client.get("/api/products")
```

```bash
# tox.ini
[tox]
envlist = py39, py310, py311

[testenv]
commands = pytest tests/
```

### tox Use Cases
- Test compatibility across Python versions.
- Run linters, formatters, and tests in isolated environments.
- Standardize CI and local test execution.

---

## 11. Tests and CI Integration

### Key CI Practices

- Run tests on every push/PR (GitHub Actions, GitLab CI, Jenkins).
- Generate **JUnit XML reports** for CI dashboards.
- Publish **coverage artifacts**.
- Send **notifications** on failure (Slack, email).

```yaml
# GitHub Actions example
- name: Run tests
  run: pytest tests/ --junitxml=report.xml --cov=myapp --cov-report=xml

- name: Upload coverage
  uses: actions/upload-artifact@v3
  with:
    name: coverage-report
    path: coverage.xml
```

```bash
# Generate HTML coverage report
pytest --cov=myapp --cov-report=html
```

CI pipeline should fail fast — run unit tests first, integration tests after.

---

## 12. Setup Testing Infrastructure

### Docker for Isolated Services

Use Docker to spin up real dependencies (databases, queues) for integration tests.

```yaml
# docker-compose.test.yml
services:
  db:
    image: postgres:15
    environment:
      POSTGRES_DB: testdb
      POSTGRES_USER: user
      POSTGRES_PASSWORD: pass
    ports:
      - "5432:5432"

  redis:
    image: redis:7
    ports:
      - "6379:6379"
```

```bash
docker-compose -f docker-compose.test.yml up -d
pytest tests/integration/
docker-compose -f docker-compose.test.yml down
```

### pytest with Docker
Use `pytest-docker` plugin to manage container lifecycle within test sessions automatically.

---

## 13. Test Coverage as a Metric

**Coverage** measures which lines/branches are executed during tests.

```bash
pytest --cov=myapp --cov-report=term-missing
```

### Types of Coverage

| Type | Measures |
|---|---|
| **Line coverage** | Were these lines executed? |
| **Branch coverage** | Were all `if/else` paths taken? |
| **Function coverage** | Were all functions called? |

### Interpreting Coverage

- **High coverage ≠ good tests**. You can have 100% coverage with meaningless assertions.
- Coverage is useful to find **untested areas**, not to prove correctness.
- Common team targets: **80–90%** for business logic; don't chase 100% blindly.
- Exclude boilerplate (migrations, configs) from coverage reports.

```ini
# .coveragerc
[run]
omit = */migrations/*, */settings.py
```

---

## 14. Advanced Integration Test Cases

### Testing Storage (Database)
- Use a **real test database** (via Docker) — avoid mocking ORM calls.
- Wrap each test in a **transaction and rollback**.
- Use factories (`factory_boy`) instead of raw fixtures.

```python
import pytest
from myapp.models import User

@pytest.fixture(autouse=True)
def db_rollback(db_session):
    yield
    db_session.rollback()

def test_create_user(db_session):
    user = User(name="Alice")
    db_session.add(user)
    db_session.commit()
    assert db_session.query(User).count() == 1
```

### Testing Message Bus (Kafka, RabbitMQ)
- Use in-memory brokers or Docker containers for tests.
- Libraries: `pytest-celery`, `fakeredis`, or real broker via Docker.
- Test **producer → consumer** contracts, not just producer output.

### Testing 3rd Party APIs
- **Don't call real APIs in tests** — they are slow, flaky, and have rate limits.
- Use `responses` or `httpretty` to intercept and mock HTTP calls.
- Use **contract testing** (Pact) to validate both sides of the API contract.

```python
import responses

@responses.activate
def test_fetch_weather():
    responses.add(responses.GET, "https://api.weather.com/today",
                  json={"temp": 22}, status=200)
    result = get_weather()
    assert result["temp"] == 22
```

### Testing Microservices — API Backward Compatibility
- Use **Pact** (consumer-driven contract testing) to ensure API changes don't break consumers.
- Test against **published API schemas** (OpenAPI/Swagger validation).
- Maintain **versioned API contracts** and run compatibility checks in CI.

```bash
# Pact flow
# 1. Consumer writes a pact (expected request/response)
# 2. Provider verifies pact on their side
# 3. Pact Broker stores and tracks compatibility
```

### Testing Microservices — General Strategies
- Use **TestContainers** for spinning up dependent services.
- Test at the **HTTP boundary** (not internal implementation).
- Validate **response schemas** and **status codes** — not just happy paths.
- Use **WireMock** or similar tools to simulate downstream services.
